# Phase 1 Final Manifest-Level Leakage Audit

Notebook fix marker: `phase1_final_leakage_audit_v2_latest_feature_source_20260421`

Purpose: record the final manifest-level leakage audit from the reviewed final LOSO split manifest and final feature schema/provenance manifest. This notebook checks that every planned fit stage uses training subjects only and that test-time privileged/teacher outputs are not allowed.

Scientific integrity rule: this audit does **not** train models, inspect final comparator runtime logs, compute metrics, run controls/calibration/influence, or open claims. It is not evidence of decoder efficacy or privileged-transfer efficacy.

## Technical Basis And Boundary

This notebook follows the repository contract:

- `configs/phase1/final_leakage_audit.json`: manifest-level leakage policy and non-claim boundary.
- `configs/phase1/final_split_feature_leakage.json`: final leakage audit is required before final comparator outputs are claim-evaluable.
- `final_split_manifest.json`: defines LOSO folds and train/test subjects.
- `final_feature_manifest.json`: defines feature schema/provenance only and must not contain feature matrix, model outputs or metrics.
- `src/phase1/final_leakage_audit.py`: CLI runner that records per-fold/per-stage fit and transform scopes.

Allowed interpretation: a manifest-level leakage audit exists for planned final split/feature scopes. It does not prove that future final comparator runtime logs are leak-free; those logs remain a separate blocker until final runners execute.

In [ ]:
# Cell 1 - Runtime setup, Drive mount, clone/pull repo, and unit tests.

from pathlib import Path
import base64
import getpass
import json
import subprocess

try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/BrianNguyen29/eeg-ds004752.git'
REPO_DIR = Path('/content/eeg-ds004752') if IN_COLAB else Path.cwd()
DRIVE_ROOT = Path('/content/drive/MyDrive/eeg-ds004752') if IN_COLAB else Path.cwd()

def run(cmd, cwd=None, check=True):
    display = []
    for item in map(str, cmd):
        display.append('<redacted>' if 'Authorization: Basic' in item else item)
    print('$ ' + ' '.join(display))
    result = subprocess.run(list(map(str, cmd)), cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if result.stdout:
        print(result.stdout)
    if check and result.returncode != 0:
        raise subprocess.CalledProcessError(result.returncode, cmd, output=result.stdout)
    return result

if IN_COLAB and not REPO_DIR.exists():
    token = getpass.getpass('Nhap GitHub token cho repo private: ')
    auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    run(['git', '-c', f'http.extraHeader=Authorization: Basic {auth}', 'clone', REPO_URL, REPO_DIR])
elif REPO_DIR.exists():
    print(REPO_DIR)

run(['git', 'pull', '--ff-only'], cwd=REPO_DIR)
run(['git', 'log', '--oneline', '-5'], cwd=REPO_DIR)
unit_result = run(['python', '-m', 'unittest', 'discover', '-s', 'tests'], cwd=REPO_DIR, check=False)
if unit_result.returncode != 0:
    print('Unit tests failed. Do not generate or review final leakage audit artifacts until tests pass.')
    raise subprocess.CalledProcessError(unit_result.returncode, ['python', '-m', 'unittest', 'discover', '-s', 'tests'])

In [ ]:
# Cell 2 - Fixed paths and expected locked prereg identity.
# Latest pointers are used so this audit follows a regenerated final feature manifest.

EXPECTED_PREREG_HASH = '87e928ea747099c336a32121bc156655a1a160d666a251c7ac41228efba96af6'
PREREG_BUNDLE = DRIVE_ROOT / 'artifacts/prereg/20260418T161442014597Z/prereg_bundle.json'
def latest_run(root, fallback):
    latest = root / 'latest.txt'
    return Path(latest.read_text(encoding='utf-8').strip()) if latest.exists() else fallback

FINAL_SPLIT_RUN = latest_run(DRIVE_ROOT / 'artifacts/phase1_final_split_manifest', DRIVE_ROOT / 'artifacts/phase1_final_split_manifest/20260421T082928310517Z')
FINAL_FEATURE_RUN = latest_run(DRIVE_ROOT / 'artifacts/phase1_final_feature_manifest', DRIVE_ROOT / 'artifacts/phase1_final_feature_manifest/20260421T084604606845Z')
OUTPUT_ROOT = DRIVE_ROOT / 'artifacts/phase1_final_leakage_audit'

print('Prereg bundle:', PREREG_BUNDLE)
print('Final split run:', FINAL_SPLIT_RUN)
print('Final feature run:', FINAL_FEATURE_RUN)
print('Output root:', OUTPUT_ROOT)

In [ ]:
# Cell 3 - Validate prereg, final split, and final feature sources.

import hashlib


def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

assert PREREG_BUNDLE.exists(), f'Missing prereg bundle: {PREREG_BUNDLE}'
bundle = read_json(PREREG_BUNDLE)
actual_prereg_hash = bundle.get('prereg_bundle_hash_sha256')
print('Prereg status:', bundle.get('status'))
print('Locked prereg identity hash:', actual_prereg_hash)
assert bundle.get('status') == 'locked', 'Prereg bundle must be locked.'
assert actual_prereg_hash == EXPECTED_PREREG_HASH, 'Prereg identity hash mismatch.'

split_summary = read_json(FINAL_SPLIT_RUN / 'phase1_final_split_manifest_summary.json')
split_validation = read_json(FINAL_SPLIT_RUN / 'phase1_final_split_manifest_validation.json')
split_manifest = read_json(FINAL_SPLIT_RUN / 'final_split_manifest.json')
feature_summary = read_json(FINAL_FEATURE_RUN / 'phase1_final_feature_manifest_summary.json')
feature_validation = read_json(FINAL_FEATURE_RUN / 'phase1_final_feature_manifest_validation.json')
feature_manifest = read_json(FINAL_FEATURE_RUN / 'final_feature_manifest.json')

print('Split status:', split_summary.get('status'), split_summary.get('split_manifest_ready'))
print('Split validation:', split_validation.get('status'))
print('Feature status:', feature_summary.get('status'), feature_summary.get('feature_manifest_ready'))
print('Feature validation:', feature_validation.get('status'))
print('Feature boundary:', {
    'contains_feature_matrix': feature_manifest.get('contains_feature_matrix'),
    'contains_model_outputs': feature_manifest.get('contains_model_outputs'),
    'contains_metrics': feature_manifest.get('contains_metrics'),
})
assert split_summary.get('status') == 'phase1_final_split_manifest_recorded'
assert split_summary.get('split_manifest_ready') is True
assert split_validation.get('status') == 'phase1_final_split_manifest_validation_passed'
assert split_validation.get('no_subject_overlap_between_train_and_test') is True
assert feature_summary.get('status') == 'phase1_final_feature_manifest_recorded'
assert feature_summary.get('feature_manifest_ready') is True
assert feature_validation.get('status') == 'phase1_final_feature_manifest_validation_passed'
assert feature_manifest.get('contains_feature_matrix') is False
assert feature_manifest.get('contains_model_outputs') is False
assert feature_manifest.get('contains_metrics') is False
assert feature_manifest.get('claim_ready') is False

In [ ]:
# Cell 4 - Source hash inventory for reproducibility.

HASH_TARGETS = [
    'configs/phase1/final_leakage_audit.json',
    'configs/phase1/final_split_feature_leakage.json',
    'src/phase1/final_leakage_audit.py',
    'src/phase1/final_split_manifest.py',
    'src/phase1/final_feature_manifest.py',
    'src/cli.py',
    'tests/unit/test_phase1_final_leakage_audit.py',
    'notebooks/21_colab_phase1_final_leakage_audit.ipynb',
]
source_hashes = {}
for rel in HASH_TARGETS:
    path = REPO_DIR / rel
    assert path.exists(), f'Missing source target: {rel}'
    source_hashes[rel] = hashlib.sha256(path.read_bytes()).hexdigest()
print(json.dumps(source_hashes, indent=2))

In [ ]:
# Cell 5 - Run the CLI-backed final manifest-level leakage audit.

cmd = [
    'python', '-m', 'src.cli', 'phase1_final_leakage_audit',
    '--profile', 't4_safe',
    '--config', str(PREREG_BUNDLE),
    '--final-split-run', str(FINAL_SPLIT_RUN),
    '--final-feature-run', str(FINAL_FEATURE_RUN),
    '--output-root', str(OUTPUT_ROOT),
    '--audit-config', 'configs/phase1/final_leakage_audit.json',
    '--readiness-config', 'configs/phase1/final_split_feature_leakage.json',
]
run(cmd, cwd=REPO_DIR)
print('Final leakage audit command completed. Review artifacts before downstream use.')

In [ ]:
# Cell 6 - Inspect latest output and verify required artifacts.

latest = OUTPUT_ROOT / 'latest.txt'
print('latest exists:', latest.exists())
assert latest.exists(), 'No latest.txt written for final leakage audit output.'
run_dir = Path(latest.read_text(encoding='utf-8').strip())
print('Latest final leakage audit output:', run_dir)
assert run_dir.exists(), f'Output run directory does not exist: {run_dir}'

required = [
    'phase1_final_leakage_audit_inputs.json',
    'phase1_final_leakage_audit_summary.json',
    'phase1_final_leakage_audit_report.md',
    'phase1_final_leakage_audit_source_links.json',
    'phase1_final_leakage_audit_input_validation.json',
    'phase1_final_leakage_audit_validation.json',
    'phase1_final_leakage_audit_claim_state.json',
    'final_leakage_audit.json',
]
for name in required:
    path = run_dir / name
    print(name, 'exists =', path.exists())
    assert path.exists(), f'Missing required artifact: {name}'

summary = read_json(run_dir / 'phase1_final_leakage_audit_summary.json')
input_validation = read_json(run_dir / 'phase1_final_leakage_audit_input_validation.json')
validation = read_json(run_dir / 'phase1_final_leakage_audit_validation.json')
claim_state = read_json(run_dir / 'phase1_final_leakage_audit_claim_state.json')
audit = read_json(run_dir / 'final_leakage_audit.json')
print(json.dumps({
    'run_dir': str(run_dir),
    'summary_status': summary.get('status'),
    'leakage_audit_ready': summary.get('leakage_audit_ready'),
    'claim_ready': summary.get('claim_ready'),
    'headline_phase1_claim_open': summary.get('headline_phase1_claim_open'),
    'n_folds': summary.get('n_folds'),
    'n_stages': summary.get('n_stages'),
    'outer_test_subject_used_in_any_fit': summary.get('outer_test_subject_used_in_any_fit'),
    'test_time_privileged_or_teacher_outputs_allowed': summary.get('test_time_privileged_or_teacher_outputs_allowed'),
    'runtime_comparator_logs_audited': summary.get('runtime_comparator_logs_audited'),
    'leakage_blockers': summary.get('leakage_blockers'),
    'claim_blockers': summary.get('claim_blockers'),
}, indent=2))

In [ ]:
# Cell 7 - Leakage-specific assertions.

assert summary.get('status') == 'phase1_final_leakage_audit_recorded'
assert summary.get('leakage_audit_ready') is True
assert summary.get('claim_ready') is False
assert summary.get('headline_phase1_claim_open') is False
assert summary.get('full_phase1_claim_bearing_run_allowed') is False
assert summary.get('outer_test_subject_used_in_any_fit') is False
assert summary.get('test_time_privileged_or_teacher_outputs_allowed') is False
assert summary.get('runtime_comparator_logs_audited') is False
assert summary.get('contains_model_outputs') is False
assert summary.get('contains_metrics') is False
assert input_validation.get('status') == 'phase1_final_leakage_audit_input_validation_passed'
assert validation.get('status') == 'phase1_final_leakage_audit_validation_passed'
assert validation.get('blockers') == []
assert audit.get('claim_ready') is False
assert audit.get('standalone_claim_ready') is False
assert audit.get('runtime_comparator_logs_audited') is False
assert audit.get('contains_model_outputs') is False
assert audit.get('contains_metrics') is False

for fold in audit['folds']:
    outer = fold['outer_test_subject']
    assert fold['no_outer_test_subject_in_any_fit'] is True
    assert fold['test_time_privileged_or_teacher_outputs_allowed'] is False
    for stage in fold['stages']:
        assert outer not in stage['fit_subjects'], f"Outer subject leaked into fit: {fold['fold_id']} {stage['stage']}"
        assert stage['fit_subjects_recorded'] is True
        assert stage['transform_subjects_recorded'] is True
        if stage['stage'] in {'teacher', 'privileged'}:
            assert stage['transform_subjects'] == fold['train_subjects']
print('Leakage review passed: no outer-test subject appears in any planned fit scope.')

In [ ]:
# Cell 8 - Write a non-claim review note.

from datetime import datetime, timezone

review_note = {
    'status': 'phase1_final_leakage_audit_review_pass_non_claim_manifest_level',
    'created_utc': datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'),
    'reviewed_run': str(run_dir),
    'scope': 'Phase 1 final manifest-level leakage audit only',
    'checks_passed': [
        'required_artifacts_present',
        'input_validation_passed',
        'leakage_validation_passed',
        'outer_test_subject_not_used_in_any_planned_fit',
        'fit_subjects_recorded_per_stage',
        'transform_subjects_recorded_per_stage',
        'test_time_privileged_or_teacher_outputs_not_allowed',
        'runtime_comparator_logs_not_claimed_as_audited',
        'contains_model_outputs_false',
        'contains_metrics_false',
        'claim_ready_false',
        'headline_phase1_claim_open_false',
    ],
    'n_folds': summary.get('n_folds'),
    'n_stages': summary.get('n_stages'),
    'leakage_audit_ready': summary.get('leakage_audit_ready'),
    'runtime_comparator_logs_audited': summary.get('runtime_comparator_logs_audited'),
    'leakage_blockers': summary.get('leakage_blockers'),
    'claim_blockers': summary.get('claim_blockers'),
    'allowed_interpretation': claim_state.get('allowed_interpretation'),
    'not_ok_to_claim': claim_state.get('not_ok_to_claim'),
    'next_allowed_steps': [
        'Use this audit as a manifest-level input contract for final comparator runner implementation.',
        'Do not treat it as audit of runtime comparator logs until final runners execute and write fold logs.',
        'Do not open headline claims until final comparator outputs, controls, calibration, influence and reporting are complete.',
    ],
}
review_path = run_dir / 'phase1_final_leakage_audit_review_note.json'
review_path.write_text(json.dumps(review_note, indent=2), encoding='utf-8')
print('Review note written:', review_path)
print(json.dumps(review_note, indent=2))

In [ ]:
# Cell 9 - Closeout.

print('================ Phase 1 Final Manifest-Level Leakage Audit Closeout ================')
print('Notebook fix marker: phase1_final_leakage_audit_v2_latest_feature_source_20260421')
print('Latest leakage audit output:', run_dir)
print('Leakage audit ready:', summary.get('leakage_audit_ready'))
print('Folds:', summary.get('n_folds'))
print('Stages:', summary.get('n_stages'))
print('Outer-test subject used in any fit:', summary.get('outer_test_subject_used_in_any_fit'))
print('Test-time privileged/teacher outputs allowed:', summary.get('test_time_privileged_or_teacher_outputs_allowed'))
print('Runtime comparator logs audited:', summary.get('runtime_comparator_logs_audited'))
print('Leakage blockers:', summary.get('leakage_blockers'))
print('Claim blockers:', summary.get('claim_blockers'))
print('')
print('CHECK REQUIRED: final_leakage_audit.json exists as manifest-level audit only. It does not audit final comparator runtime logs.')
print('')
print('NOT OK TO CLAIM: This leakage audit notebook does not prove decoder efficacy, A3/A4 efficacy, A4 superiority, privileged-transfer efficacy, or full Phase 1 neural comparator performance.')